# Test 10: Training Speed — Chaos Activation vs ReLU

**Гипотеза:** Chaos activation (sin(8x)+0.5·tanh(4x)) медленнее ReLU за счёт вычисления двух трансцендентных функций вместо одной. Однако удвоение нейронов для компенсации мёртвых нейронов при ReLU обходится дороже.

**План:**
- 4 архитектуры: V4 Chaos (256→128), TopK+ReLU (256→128), TopK+ReLU 2x (512→256), Dense+ReLU 2x (512→256)
- Замер: время на эпоху, val_loss, dead neurons
- N=5 прогонов, MNIST 60K
- Метрики: время обучения, качество, мёртвые нейроны

**Зачем:**
- Ответ на вопрос: «Может просто удвоить нейроны с ReLU вместо кастомной активации?»
- Сравнение стоимости вычислений vs стоимости компенсации мёртвых нейронов

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from scipy import stats
import json
import time
from datetime import datetime

print(f"TF version: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

/Users/savenkovviktor/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TF version: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
@tf.function
def chaos_activation(x):
    return tf.sin(8.0 * x) + 0.5 * tf.tanh(4.0 * x)


class KSparseLayer(layers.Layer):
    def __init__(self, k=32, **kwargs):
        super().__init__(**kwargs)
        self.k = k

    def call(self, inputs, training=None):
        latent_dim = tf.shape(inputs)[1]
        _, indices = tf.nn.top_k(tf.abs(inputs), k=self.k, sorted=False)
        mask = tf.reduce_sum(
            tf.one_hot(indices, latent_dim, dtype=inputs.dtype), axis=1
        )
        return inputs * mask

    def get_config(self):
        config = super().get_config()
        config.update({"k": self.k})
        return config


def analyze_latent(encoder, images, threshold=1e-6):
    latents = encoder.predict(images, verbose=0)
    variance_per_dim = np.var(latents, axis=0)
    dead_mask = np.all(np.abs(latents) < threshold, axis=0)
    zero_mask = np.abs(latents) < threshold
    return {
        'mean_variance': float(np.mean(variance_per_dim)),
        'dead_neurons': int(np.sum(dead_mask)),
        'total_neurons': latents.shape[1],
        'sparsity': float(np.mean(zero_mask)),
    }


print("Core components ready.")

Core components ready.


In [3]:
# === Архитектуры ===

def build_v4_chaos(image_size=(28, 28), latent_dim=128, k_active=32):
    """V4: Chaos activation, стандартный размер (256->128)"""
    input_img = keras.Input(shape=(*image_size, 1))
    x = layers.Flatten()(input_img)
    x = layers.Dense(256)(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(latent_dim, name='latent_pre')(x)
    x = layers.Activation(chaos_activation)(x)
    latent = KSparseLayer(k=k_active, name='latent_ksparse')(x)
    encoder = keras.Model(input_img, latent, name='encoder')

    x = layers.Dense(256)(latent)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dropout(0.1)(x)
    decoded = layers.Dense(np.prod(image_size), activation='sigmoid')(x)
    decoded = layers.Reshape((*image_size, 1))(decoded)

    autoencoder = keras.Model(input_img, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder, encoder


def build_topk_relu(image_size=(28, 28), latent_dim=128, k_active=32, hidden=256):
    """TopK+ReLU с настраиваемым размером скрытого слоя"""
    input_img = keras.Input(shape=(*image_size, 1))
    x = layers.Flatten()(input_img)
    x = layers.Dense(hidden, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(latent_dim, activation='relu', name='latent_pre')(x)
    latent = KSparseLayer(k=k_active, name='latent_ksparse')(x)
    encoder = keras.Model(input_img, latent, name='encoder')

    x = layers.Dense(hidden, activation='relu')(latent)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.1)(x)
    decoded = layers.Dense(np.prod(image_size), activation='sigmoid')(x)
    decoded = layers.Reshape((*image_size, 1))(decoded)

    autoencoder = keras.Model(input_img, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder, encoder


def build_dense_relu(image_size=(28, 28), latent_dim=128, hidden=256):
    """Dense+ReLU (без top-k) с настраиваемым размером"""
    h, w = image_size
    input_img = keras.Input(shape=(h, w, 1))
    x = layers.Flatten()(input_img)
    x = layers.Dense(hidden, activation='relu')(x)
    x = layers.Dense(hidden // 2, activation='relu')(x)
    latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)
    encoder = keras.Model(input_img, latent, name='encoder')

    x = layers.Dense(hidden // 2, activation='relu')(latent)
    x = layers.Dense(hidden, activation='relu')(x)
    decoded = layers.Dense(h * w, activation='sigmoid')(x)
    decoded = layers.Reshape((h, w, 1))(decoded)

    autoencoder = keras.Model(input_img, decoded)
    autoencoder.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')
    return autoencoder, encoder


print("Architectures defined.")

Architectures defined.


In [4]:
# === Данные ===
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print(f"MNIST train: {x_train.shape}, test: {x_test.shape}")

MNIST train: (60000, 28, 28, 1), test: (10000, 28, 28, 1)


In [ ]:
# === Эксперимент ===
NUM_RUNS = 5
EPOCHS = 10
BATCH_SIZE = 128
LATENT_DIM = 128
K_ACTIVE = 32

architectures = {
    'V4_Chaos_256':       lambda: build_v4_chaos(latent_dim=LATENT_DIM, k_active=K_ACTIVE),
    'TopK_ReLU_256':      lambda: build_topk_relu(latent_dim=LATENT_DIM, k_active=K_ACTIVE, hidden=256),
    'TopK_ReLU_512':      lambda: build_topk_relu(latent_dim=LATENT_DIM * 2, k_active=K_ACTIVE, hidden=512),
    'Dense_ReLU_256':     lambda: build_dense_relu(latent_dim=LATENT_DIM, hidden=256),
    'Dense_ReLU_512':     lambda: build_dense_relu(latent_dim=LATENT_DIM * 2, hidden=512),
}

results = {}

for arch_name, builder in architectures.items():
    runs = []
    for run in range(NUM_RUNS):
        np.random.seed(run)
        tf.random.set_seed(run)

        ae, enc = builder()
        n_params = ae.count_params()

        # Прогрев (первая эпоха компилирует граф)
        ae.fit(x_train, x_train, epochs=1, batch_size=BATCH_SIZE, verbose=0)

        # Замер
        t0 = time.time()
        history = ae.fit(
            x_train, x_train,
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            validation_data=(x_test, x_test),
            verbose=0
        )
        train_time = time.time() - t0

        s = analyze_latent(enc, x_test)
        s['val_loss'] = history.history['val_loss'][-1]
        s['train_time'] = train_time
        s['time_per_epoch'] = train_time / EPOCHS
        s['n_params'] = n_params
        runs.append(s)

        print(f"  {arch_name:20s} run {run+1}/{NUM_RUNS}: "
              f"time={train_time:.2f}s ({train_time/EPOCHS:.2f}s/ep), "
              f"loss={s['val_loss']:.4f}, "
              f"dead={s['dead_neurons']}/{s['total_neurons']}, "
              f"params={n_params:,}")

        del ae, enc
        keras.backend.clear_session()

    results[arch_name] = runs
    print()

print("All experiments done.")

2026-04-06 12:35:55.924672: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2026-04-06 12:35:55.924694: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2026-04-06 12:35:55.924697: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
2026-04-06 12:35:55.924713: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-06 12:35:55.924720: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-04-06 12:35:56.385158: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


  V4_Chaos_256         run 1/5: time=76.03s (7.60s/ep), loss=0.0669, dead=0/128, params=469,392
  V4_Chaos_256         run 2/5: time=74.90s (7.49s/ep), loss=0.0672, dead=0/128, params=469,392
  V4_Chaos_256         run 3/5: time=76.19s (7.62s/ep), loss=0.0670, dead=0/128, params=469,392
  V4_Chaos_256         run 4/5: time=75.28s (7.53s/ep), loss=0.0667, dead=0/128, params=469,392
  V4_Chaos_256         run 5/5: time=76.33s (7.63s/ep), loss=0.0671, dead=0/128, params=469,392

  TopK_ReLU_256        run 1/5: time=62.05s (6.21s/ep), loss=0.0117, dead=70/128, params=469,392
  TopK_ReLU_256        run 2/5: time=58.79s (5.88s/ep), loss=0.0125, dead=75/128, params=469,392
  TopK_ReLU_256        run 3/5: time=62.70s (6.27s/ep), loss=0.0114, dead=65/128, params=469,392
  TopK_ReLU_256        run 4/5: time=62.89s (6.29s/ep), loss=0.0120, dead=66/128, params=469,392
  TopK_ReLU_256        run 5/5: time=60.87s (6.09s/ep), loss=0.0108, dead=60/128, params=469,392

  TopK_ReLU_512        run 1/5: t

In [ ]:
# === Сводная таблица ===
print(f"{'Architecture':<20} {'Time/ep':>10} {'Val Loss':>12} {'Dead':>10} {'Params':>12}")
print('-' * 70)

summary = {}

for arch_name in architectures:
    runs = results[arch_name]
    t = [r['time_per_epoch'] for r in runs]
    l = [r['val_loss'] for r in runs]
    d = [r['dead_neurons'] for r in runs]
    p = runs[0]['n_params']

    row = {
        'time_mean': np.mean(t), 'time_std': np.std(t),
        'loss_mean': np.mean(l), 'loss_std': np.std(l),
        'dead_mean': np.mean(d), 'dead_std': np.std(d),
        'n_params': p,
        'times': t, 'losses': l, 'deads': d,
    }
    summary[arch_name] = row

    print(f"{arch_name:<20} "
          f"{row['time_mean']:.2f}s ± {row['time_std']:.2f} "
          f"{row['loss_mean']:.4f} ± {row['loss_std']:.4f} "
          f"{row['dead_mean']:>5.1f} ± {row['dead_std']:.1f} "
          f"{row['n_params']:>10,}")

In [ ]:
# === Статистические сравнения ===
print("STATISTICAL COMPARISONS (Welch t-test)")
print('=' * 70)

comparisons = [
    ('V4_Chaos_256', 'TopK_ReLU_256', 'Chaos vs ReLU (одинаковый размер)'),
    ('V4_Chaos_256', 'TopK_ReLU_512', 'Chaos vs ReLU 2x (удвоенный)'),
    ('V4_Chaos_256', 'Dense_ReLU_512', 'Chaos vs Dense ReLU 2x'),
    ('TopK_ReLU_256', 'TopK_ReLU_512', 'ReLU 1x vs ReLU 2x'),
]

for a1, a2, label in comparisons:
    t1 = summary[a1]['times']
    t2 = summary[a2]['times']
    l1 = summary[a1]['losses']
    l2 = summary[a2]['losses']

    t_time, p_time = stats.ttest_ind(t1, t2, equal_var=False)
    t_loss, p_loss = stats.ttest_ind(l1, l2, equal_var=False)

    print(f"\n  {label}:")
    print(f"    Время/эпоха: {np.mean(t1):.2f}s vs {np.mean(t2):.2f}s "
          f"(разница {((np.mean(t2)/np.mean(t1))-1)*100:+.1f}%, p={p_time:.4f})")
    print(f"    Val loss:    {np.mean(l1):.4f} vs {np.mean(l2):.4f} "
          f"(p={p_loss:.4f})")
    print(f"    Dead:        {summary[a1]['dead_mean']:.1f} vs {summary[a2]['dead_mean']:.1f}")
    print(f"    Params:      {summary[a1]['n_params']:,} vs {summary[a2]['n_params']:,}")

In [ ]:
# === Визуализация ===
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

arch_names = list(architectures.keys())
x_pos = np.arange(len(arch_names))
colors = ['#2196F3', '#FF9800', '#FF5722', '#4CAF50', '#8BC34A']

# Время на эпоху
means = [summary[a]['time_mean'] for a in arch_names]
stds = [summary[a]['time_std'] for a in arch_names]
axes[0].bar(x_pos, means, yerr=stds, capsize=5, color=colors, alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([n.replace('_', '\n') for n in arch_names], fontsize=7)
axes[0].set_ylabel('Секунды')
axes[0].set_title('Время на эпоху')
axes[0].grid(True, alpha=0.3, axis='y')

# Val loss
means = [summary[a]['loss_mean'] for a in arch_names]
stds = [summary[a]['loss_std'] for a in arch_names]
axes[1].bar(x_pos, means, yerr=stds, capsize=5, color=colors, alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([n.replace('_', '\n') for n in arch_names], fontsize=7)
axes[1].set_ylabel('MSE')
axes[1].set_title('Val Loss')
axes[1].grid(True, alpha=0.3, axis='y')

# Dead neurons
means = [summary[a]['dead_mean'] for a in arch_names]
stds = [summary[a]['dead_std'] for a in arch_names]
axes[2].bar(x_pos, means, yerr=stds, capsize=5, color=colors, alpha=0.8)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels([n.replace('_', '\n') for n in arch_names], fontsize=7)
axes[2].set_ylabel('Dead Neurons')
axes[2].set_title('Мёртвые нейроны')
axes[2].grid(True, alpha=0.3, axis='y')

# Параметры
params = [summary[a]['n_params'] for a in arch_names]
axes[3].bar(x_pos, params, color=colors, alpha=0.8)
axes[3].set_xticks(x_pos)
axes[3].set_xticklabels([n.replace('_', '\n') for n in arch_names], fontsize=7)
axes[3].set_ylabel('Параметры')
axes[3].set_title('Количество параметров')
axes[3].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Test 10: Training Speed — Chaos vs ReLU vs ReLU 2x (N={NUM_RUNS} runs)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/training_speed_test.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: images/training_speed_test.png")

In [ ]:
# === Эффективность: quality per second ===
print("\nЭффективность (качество за единицу времени):")
print(f"{'Architecture':<20} {'Loss':>8} {'Time':>8} {'Dead':>6} {'Eff (1/loss/time)':>18}")
print('-' * 65)

for arch_name in architectures:
    s = summary[arch_name]
    eff = 1.0 / (s['loss_mean'] * s['time_mean'])
    print(f"{arch_name:<20} "
          f"{s['loss_mean']:>8.4f} "
          f"{s['time_mean']:>7.2f}s "
          f"{s['dead_mean']:>5.1f} "
          f"{eff:>18.2f}")

In [ ]:
# === Сохранение результатов ===
save_data = {
    'experiment': 'training_speed_comparison',
    'timestamp': datetime.now().isoformat(),
    'config': {
        'dataset': 'MNIST',
        'num_runs': NUM_RUNS,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'latent_dim': LATENT_DIM,
        'k_active': K_ACTIVE,
    },
    'summary': {
        arch: {k: v for k, v in vals.items() if k not in ('times', 'losses', 'deads')}
        for arch, vals in summary.items()
    },
    'raw_results': {
        arch: runs for arch, runs in results.items()
    },
}

json_path = f'../jsons/training_speed_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(json_path, 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print(f"Results saved: {json_path}")